In [ ]:
from pathlib import Path
from shutil import copytree

source = Path("/kaggle/input/datasets/cphmlhng/docling")
target = Path("/kaggle/working/")
copytree(source, target, dirs_exist_ok=True)

In [ ]:
# %pip install --no-cache-dir "/kaggle/working/plugin"

# %pip install --no-cache-dir "/kaggle/working/plugin[easyocr]"
# %pip uninstall -y torchaudio torchcodec

%pip install --no-cache-dir "/kaggle/working/plugin[rapidocr,doclayout-yolo,rapid-table]"
%pip uninstall -y onnxruntime torchaudio torchcodec
%pip install --no-cache-dir --force-reinstall --no-deps onnxruntime-gpu

# %pip install --no-cache-dir "/kaggle/working/plugin[rapidocr,doclayout-yolo]"
# %pip uninstall -y onnxruntime torchaudio torchcodec
# %pip install --no-cache-dir --force-reinstall --no-deps onnxruntime-gpu

# %pip install --no-cache-dir --no-deps paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/
# %pip install --no-cache-dir "/kaggle/working/plugin[rapidocr,pp-doclayout-v3]"
# %pip uninstall -y torchaudio torchcodec

# %pip install --no-cache-dir "/kaggle/working/plugin" "torch==2.6.0" "torchvision==0.21.0"
# %pip install --no-cache-dir "docling-surya==0.1.0" "surya-ocr==0.17.0" "pillow==10.4.0" "transformers==4.57.6"

In [ ]:
USE_EASYOCR = False
USE_RAPIDOCR = True
USE_SURYA_OCR = False

USE_DOCLAYOUT_YOLO = True
USE_PP_DOCLAYOUT_V3 = False

USE_TABLEFORMER_ACCURATE = False
USE_RAPID_TABLE_SLANET_PLUS = False
USE_RAPID_TABLE_UNITABLE = True

USE_DYNAMIC_SCALE = True
# False = OcrMode.DEFAULT (hybrid native text + OCR, fair end-to-end benchmark)
# True  = OcrMode.FULL_PAGE (force full-page OCR for OCR-only benchmark)
USE_FULL_PAGE_OCR = False

if sum([USE_EASYOCR, USE_RAPIDOCR, USE_SURYA_OCR]) > 1:
    raise ValueError("Enable at most one OCR engine at a time.")
if sum([USE_DOCLAYOUT_YOLO, USE_PP_DOCLAYOUT_V3]) > 1:
    raise ValueError("Enable at most one layout plugin at a time.")
if sum([USE_TABLEFORMER_ACCURATE, USE_RAPID_TABLE_SLANET_PLUS, USE_RAPID_TABLE_UNITABLE]) > 1:
    raise ValueError("Enable at most one table model at a time.")

In [ ]:
import os
from importlib.metadata import version

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["DISABLE_TQDM"] = "true"
os.environ["TRITON_DEFAULT_FP_FUSION"] = "1"
# Set HF_TOKEN via Kaggle Secrets if needed (do not hardcode tokens)

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    EasyOcrOptions,
    HeadingHierarchyOptions,
    LayoutOptions,
    OcrAutoOptions,
    OcrMode,
    PdfPipelineOptions,
    RapidOcrOptions,
    TableFormerMode,
    TableStructureOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from _doclayout_yolo import DocLayoutYoloOptions
from _pp_doclayout_v3 import PPDocLayoutV3Options
from _rapid_table import RapidTableOptions

if USE_SURYA_OCR:
    from docling.models.base_ocr_model import BaseOcrModel
    from docling_surya import SuryaOcrOptions
    from docling_surya.plugin import SuryaOcrModel

    # Compatibility shim: docling-surya 0.1.0 calls post_process_cells(cells, page)
    # without conv_res, which became required in docling>=2.150.
    if not getattr(SuryaOcrModel, "_docling_compat_patched", False):
        _orig_surya_call = SuryaOcrModel.__call__
        _orig_post_process = BaseOcrModel.post_process_cells

        def _patched_surya_call(self, conv_res, page_batch):
            self._current_conv_res = conv_res
            yield from _orig_surya_call(self, conv_res, page_batch)

        def _patched_post_process(self, ocr_cells, page, conv_res=None, *args, **kwargs):
            if conv_res is None:
                conv_res = getattr(self, "_current_conv_res", None)
            return _orig_post_process(self, ocr_cells, page, conv_res, *args, **kwargs)

        SuryaOcrModel.__call__ = _patched_surya_call
        BaseOcrModel.post_process_cells = _patched_post_process
        SuryaOcrModel._docling_compat_patched = True

print("docling:", version("docling"))
print("docling-core:", version("docling-core"))

In [ ]:
accelerator_options = AcceleratorOptions(
    device=AcceleratorDevice.CUDA,
    num_threads=4,
    cuda_use_flash_attention2=False,
)

In [ ]:
ocr_mode = OcrMode.FULL_PAGE if USE_FULL_PAGE_OCR else OcrMode.DEFAULT

if USE_EASYOCR:
    ocr_options = EasyOcrOptions(
        lang=["ja", "en"],
        mode=ocr_mode,
    )
elif USE_RAPIDOCR:
    # lang=["ch"] resolves to PP-OCRv6_medium multilingual (covers EN + JA + ZH)
    ocr_options = RapidOcrOptions(
        lang=["ch"],
        backend="onnxruntime",
        model_size="medium",
        mode=ocr_mode,
        rapidocr_params={
            "EngineConfig.onnxruntime.cuda_ep_cfg.gpu_mem_limit": 8 * 1024 * 1024 * 1024,
            "EngineConfig.onnxruntime.cuda_ep_cfg.arena_extend_strategy": "kSameAsRequested",
        },
    )
elif USE_SURYA_OCR:
    ocr_options = SuryaOcrOptions(
        lang=["en", "ja"],
        force_full_page_ocr=USE_FULL_PAGE_OCR,
    )
else:
    ocr_options = OcrAutoOptions(lang=["en", "ja"], mode=ocr_mode)

In [ ]:
if USE_DOCLAYOUT_YOLO:
    layout_options = DocLayoutYoloOptions(
        image_size=1024,
        confidence=0.2,
        device="cuda:0",
        keep_empty_clusters=True,
    )
elif USE_PP_DOCLAYOUT_V3:
    layout_options = PPDocLayoutV3Options(
        threshold=0.5,
        layout_nms=True,
        device="gpu:0",
        keep_empty_clusters=True,
    )
else:
    layout_options = LayoutOptions(
        keep_empty_clusters=True,
    )

In [ ]:
_rapid_table_engine_cfg = {
    "use_cuda": True,
    "cuda_ep_cfg": {
        "gpu_mem_limit": 8 * 1024 * 1024 * 1024,
        "arena_extend_strategy": "kSameAsRequested",
    },
}

if USE_RAPID_TABLE_SLANET_PLUS:
    table_options = RapidTableOptions(
        model_type="slanet_plus",
        scale=2.0,
        use_ocr=False,
        engine_cfg=_rapid_table_engine_cfg,
    )
elif USE_RAPID_TABLE_UNITABLE:
    table_options = RapidTableOptions(
        model_type="unitable",
        scale=2.0,
        use_ocr=False,
        engine_cfg={"use_cuda": True, "gpu_id": 0},
    )
else:
    table_options = TableStructureOptions(
        mode=TableFormerMode.ACCURATE,
        do_cell_matching=True,
    )

In [ ]:
heading_options = HeadingHierarchyOptions(
    enabled=False,
    use_bookmarks=True,
    use_numbering=True,
    use_style=True,
    use_font_style=True,
    max_level=6,
)

In [ ]:
pdf_options = PdfPipelineOptions(
    # Runtime & security
    document_timeout=180,
    accelerator_options=accelerator_options,
    enable_remote_services=False,
    allow_external_plugins=True,
    artifacts_path=None,  # Set to a local model directory for offline inference.

    # OCR, layout & table structure
    do_ocr=True,
    ocr_options=ocr_options,
    do_table_structure=True,
    table_structure_options=table_options,
    layout_options=layout_options,
    force_backend_text=False,

    # Reading order / heading hierarchy
    use_reading_order_separators=False,
    heading_hierarchy_options=heading_options,

    # Assets cho validator, crop & recovery
    images_scale=2.0,
    generate_page_images=False,
    generate_picture_images=False,
    generate_table_images=False,
    generate_parsed_pages=False,

    # Built-in enrichment
    do_picture_classification=False,
    do_picture_description=False,
    do_chart_extraction=False,
    do_code_enrichment=False,
    do_formula_enrichment=False,

    # Batching / throughput
    ocr_batch_size=2,
    layout_batch_size=2,
    table_batch_size=2,
    batch_polling_interval_seconds=0.5,
    queue_max_size=100,
    stage_shutdown_timeout_seconds=60.0,
)

In [ ]:
import json
import multiprocessing as mp
import random
import resource
import statistics
import subprocess
import threading
from queue import Empty
from time import perf_counter
import traceback

import pypdfium2 as pdfium
from docling_core.types.doc import ImageRefMode
from tqdm.auto import tqdm


def compute_dynamic_scales(
    pdf_path,
    target_layout_short_px=1024,
    target_ocr_short_px=1800,
    max_ocr_long_px=3200,
    layout_bounds=(1.0, 4.0),
    ocr_bounds=(1.5, 5.0),
):
    doc = pdfium.PdfDocument(str(pdf_path))
    try:
        if len(doc) == 0:
            return pdf_options.images_scale, ocr_options.scale
        width_pt, height_pt = doc[0].get_size()
    finally:
        doc.close()

    short_pt = max(min(width_pt, height_pt), 1.0)
    long_pt = max(width_pt, height_pt, 1.0)

    layout_scale = max(
        layout_bounds[0],
        min(layout_bounds[1], target_layout_short_px / short_pt),
    )
    raw_ocr_scale = min(
        target_ocr_short_px / short_pt,
        max_ocr_long_px / long_pt,
    )
    ocr_scale = max(ocr_bounds[0], min(ocr_bounds[1], raw_ocr_scale))
    return round(layout_scale, 2), round(ocr_scale, 2)


def apply_dynamic_scales(converter, layout_scale, ocr_scale):
    # Update only runtime stage attributes, NEVER pipeline_options,
    # so Docling's options_hash stays constant and reuses the single cached pipeline.
    for pipeline in getattr(converter, "initialized_pipelines", {}).values():
        if hasattr(pipeline, "preprocessing_model") and hasattr(
            pipeline.preprocessing_model, "options"
        ):
            pipeline.preprocessing_model.options.images_scale = layout_scale
        if hasattr(pipeline, "layout_model") and pipeline.layout_model is not None:
            if hasattr(pipeline.layout_model, "scale"):
                pipeline.layout_model.scale = layout_scale
        if hasattr(pipeline, "ocr_model") and pipeline.ocr_model is not None:
            if hasattr(pipeline.ocr_model, "scale"):
                pipeline.ocr_model.scale = ocr_scale


def build_converter():
    options = pdf_options.model_copy(deep=True)
    options.accelerator_options = AcceleratorOptions(
        device="cuda:0",
        num_threads=4,
        cuda_use_flash_attention2=False,
    )
    if USE_DOCLAYOUT_YOLO:
        options.layout_options.device = "cuda:0"
    if USE_PP_DOCLAYOUT_V3:
        options.layout_options.device = "gpu:0"

    return DocumentConverter(
        allowed_formats=[InputFormat.PDF],
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=options),
        },
    )


def _gpu_worker(gpu_index, task_queue, result_queue, model_init_lock):
    # Expose exactly one physical GPU to this process. Inside the worker,
    # that GPU is always addressed as cuda:0/gpu:0.
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
    try:
        converter = build_converter()
        with model_init_lock:
            converter.initialize_pipeline(InputFormat.PDF)
    except Exception as exc:
        result_queue.put({
            "kind": "worker_error",
            "gpu": gpu_index,
            "error": str(exc),
            "traceback": traceback.format_exc(),
        })
        return

    first_conversion_complete = False
    while True:
        task = task_queue.get()
        if task is None:
            peak_ram_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0
            result_queue.put({
                "kind": "worker_stats",
                "gpu": gpu_index,
                "peak_ram_mb": round(peak_ram_mb, 2),
            })
            return

        source, destination = map(Path, task)
        result_queue.put({
            "kind": "file_started",
            "gpu": gpu_index,
            "file": str(source),
        })
        started_at = perf_counter()
        num_pages = 0
        try:
            destination.parent.mkdir(parents=True, exist_ok=True)
            if USE_DYNAMIC_SCALE:
                layout_scale, ocr_scale = compute_dynamic_scales(source)
                apply_dynamic_scales(converter, layout_scale, ocr_scale)
            if not first_conversion_complete:
                # RapidOCR/Surya may download model weights lazily on first convert.
                # Serialize the first conversion across GPU workers to prevent race corruption.
                with model_init_lock:
                    result = converter.convert(source)
                first_conversion_complete = True
            else:
                result = converter.convert(source)
            num_pages = max(len(result.document.pages), 1)
            result.document.save_as_json(
                destination,
                image_mode=ImageRefMode.PLACEHOLDER,
                ensure_ascii=False,
            )
            error = None
        except Exception as exc:
            error = {
                "file": str(source),
                "gpu": gpu_index,
                "error_type": type(exc).__name__,
                "error": str(exc),
                "traceback": traceback.format_exc(),
            }

        result_queue.put({
            "kind": "file_result",
            "gpu": gpu_index,
            "error": error,
            "num_pages": num_pages,
            "elapsed_seconds": perf_counter() - started_at,
        })


def _start_vram_monitor(gpu_indices, poll_interval=0.5):
    stop_event = threading.Event()
    peak_vram_mb = {gpu_index: 0.0 for gpu_index in gpu_indices}

    def _poll():
        while not stop_event.is_set():
            try:
                out = subprocess.check_output(
                    [
                        "nvidia-smi",
                        "--query-gpu=index,memory.used",
                        "--format=csv,noheader,nounits",
                    ],
                    text=True,
                    timeout=2,
                )
                for line in out.strip().splitlines():
                    parts = [p.strip() for p in line.split(",")]
                    if len(parts) == 2:
                        idx = int(parts[0])
                        used_mb = float(parts[1])
                        if idx in peak_vram_mb and used_mb > peak_vram_mb[idx]:
                            peak_vram_mb[idx] = used_mb
            except Exception:
                pass
            stop_event.wait(poll_interval)

    thread = threading.Thread(target=_poll, daemon=True)
    thread.start()
    return stop_event, thread, peak_vram_mb


def _active_config_summary():
    return {
        "layout": (
            "doclayout_yolo" if USE_DOCLAYOUT_YOLO
            else "pp_doclayout_v3" if USE_PP_DOCLAYOUT_V3
            else "docling_layout_heron"
        ),
        "ocr": (
            "rapidocr_pp_ocrv6_medium" if USE_RAPIDOCR
            else "easyocr" if USE_EASYOCR
            else "surya" if USE_SURYA_OCR
            else "auto"
        ),
        "ocr_mode": "full_page" if USE_FULL_PAGE_OCR else "default",
        "table": (
            "rapid_table_unitable" if USE_RAPID_TABLE_UNITABLE
            else "rapid_table_slanet_plus" if USE_RAPID_TABLE_SLANET_PLUS
            else "tableformer_accurate"
        ),
        "dynamic_scale": USE_DYNAMIC_SCALE,
        "do_ocr": pdf_options.do_ocr,
        "do_table_structure": pdf_options.do_table_structure,
        "do_formula_enrichment": pdf_options.do_formula_enrichment,
        "do_picture_description": pdf_options.do_picture_description,
    }


def convert_dataset(name, input_dir, output_dir, limit=None, seed=None):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    pdf_files = sorted(input_dir.rglob("*.pdf"))
    if limit is not None and limit < len(pdf_files):
        if seed is not None:
            pdf_files = sorted(random.Random(seed).sample(pdf_files, limit))
        else:
            pdf_files = pdf_files[:limit]
    if not pdf_files:
        raise FileNotFoundError(f"No PDF files found in {input_dir}")

    gpu_indices = (0, 1)
    # Kaggle runs Linux. fork keeps notebook-defined functions available in children.
    # Do not create a converter or initialize CUDA in the parent before this point.
    context = mp.get_context("fork")
    task_queue = context.Queue()
    result_queue = context.Queue()
    model_init_lock = context.Lock()
    workers = [
        context.Process(
            target=_gpu_worker,
            args=(gpu_index, task_queue, result_queue, model_init_lock),
            name=f"docling-gpu-{gpu_index}",
        )
        for gpu_index in gpu_indices
    ]

    wall_started_at = perf_counter()
    for worker in workers:
        worker.start()
    vram_stop, vram_thread, peak_vram_mb = _start_vram_monitor(gpu_indices)
    for pdf_path in pdf_files:
        output_path = output_dir / pdf_path.relative_to(input_dir).with_suffix(".json")
        task_queue.put((str(pdf_path), str(output_path)))
    for _ in workers:
        task_queue.put(None)

    error_path = output_dir / "errors.json"
    current_path = output_dir / "current_files.json"
    perf_path = output_dir / "perf.json"
    errors = []
    error_path.write_text("[]", encoding="utf-8")
    passed = 0
    total_pages = 0
    elapsed_times = []
    gpu_counts = {gpu_index: 0 for gpu_index in gpu_indices}
    current_files = {gpu_index: None for gpu_index in gpu_indices}
    worker_peak_ram_mb = {}
    current_path.write_text(
        json.dumps(current_files, indent=2), encoding="utf-8"
    )
    progress = tqdm(total=len(pdf_files), desc=name, unit="file")
    try:
        while passed + len(errors) < len(pdf_files):
            try:
                outcome = result_queue.get(timeout=5)
            except Empty:
                failed_workers = [w for w in workers if w.exitcode not in (None, 0)]
                if failed_workers:
                    details = ", ".join(
                        f"{w.name}=exit {w.exitcode}" for w in failed_workers
                    )
                    raise RuntimeError(f"Docling worker stopped unexpectedly: {details}")
                if all(not w.is_alive() for w in workers):
                    raise RuntimeError("All workers exited before completing the dataset")
                continue

            if outcome["kind"] == "worker_error":
                raise RuntimeError(
                    f"GPU {outcome['gpu']} initialization failed: "
                    f"{outcome['error']}\n{outcome['traceback']}"
                )

            if outcome["kind"] == "worker_stats":
                worker_peak_ram_mb[outcome["gpu"]] = outcome["peak_ram_mb"]
                continue

            if outcome["kind"] == "file_started":
                current_files[outcome["gpu"]] = outcome["file"]
                current_path.write_text(
                    json.dumps(current_files, ensure_ascii=False, indent=2),
                    encoding="utf-8",
                )
                progress.set_postfix(
                    passed=passed,
                    failed=len(errors),
                    gpu0=gpu_counts[0],
                    gpu1=gpu_counts[1],
                    file0=(Path(current_files[0]).stem[:10] if current_files[0] else "-"),
                    file1=(Path(current_files[1]).stem[:10] if current_files[1] else "-"),
                )
                continue

            elapsed_times.append(outcome["elapsed_seconds"])
            total_pages += outcome.get("num_pages", 0)
            gpu_counts[outcome["gpu"]] += 1
            current_files[outcome["gpu"]] = None
            current_path.write_text(
                json.dumps(current_files, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )
            if outcome["error"] is None:
                passed += 1
            else:
                error = outcome["error"]
                errors.append(error)
                error_path.write_text(
                    json.dumps(errors, ensure_ascii=False, indent=2),
                    encoding="utf-8",
                )
                tqdm.write(
                    f"[FAILED][GPU {outcome['gpu']}] {error['file']}\n"
                    f"{error['error_type']}: {error['error']}"
                )

            average_seconds = sum(elapsed_times) / len(elapsed_times)
            progress.update(1)
            progress.set_postfix(
                passed=passed,
                failed=len(errors),
                gpu0=gpu_counts[0],
                gpu1=gpu_counts[1],
                file0=(Path(current_files[0]).stem[:10] if current_files[0] else "-"),
                file1=(Path(current_files[1]).stem[:10] if current_files[1] else "-"),
                avg_time=f"{average_seconds:.2f}s",
            )

        # Drain final worker_stats messages before joining workers
        drain_deadline = perf_counter() + 10.0
        while len(worker_peak_ram_mb) < len(workers) and perf_counter() < drain_deadline:
            try:
                outcome = result_queue.get(timeout=0.5)
                if outcome["kind"] == "worker_stats":
                    worker_peak_ram_mb[outcome["gpu"]] = outcome["peak_ram_mb"]
            except Empty:
                if all(not w.is_alive() for w in workers):
                    break
    finally:
        vram_stop.set()
        vram_thread.join(timeout=3)
        progress.close()
        for worker in workers:
            worker.join(timeout=30)
        for worker in workers:
            if worker.is_alive():
                worker.terminate()
                worker.join()

    error_path.write_text(
        json.dumps(errors, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    wall_seconds = perf_counter() - wall_started_at
    sorted_times = sorted(elapsed_times)
    average_seconds = sum(elapsed_times) / len(elapsed_times)
    median_seconds = statistics.median(sorted_times)
    p95_seconds = sorted_times[min(int(len(sorted_times) * 0.95), len(sorted_times) - 1)]
    avg_seconds_per_page = (sum(elapsed_times) / total_pages) if total_pages > 0 else None
    effective_seconds_per_page = (wall_seconds / total_pages) if total_pages > 0 else None
    throughput_files = len(elapsed_times) / wall_seconds
    throughput_pages = (total_pages / wall_seconds) if total_pages > 0 else 0.0
    parent_peak_ram_mb = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0, 2)
    max_worker_ram_mb = max(worker_peak_ram_mb.values()) if worker_peak_ram_mb else None
    max_gpu_vram_mb = max(peak_vram_mb.values()) if peak_vram_mb else None

    perf = {
        "dataset": name,
        "config": _active_config_summary(),
        "num_gpus": len(gpu_indices),
        "total_files": len(pdf_files),
        "passed": passed,
        "failed": len(errors),
        "total_pages": total_pages,
        "gpu_counts": {str(k): v for k, v in gpu_counts.items()},
        "wall_seconds": round(wall_seconds, 2),
        "avg_seconds_per_file": round(average_seconds, 4),
        "median_seconds_per_file": round(median_seconds, 4),
        "p95_seconds_per_file": round(p95_seconds, 4),
        "avg_seconds_per_page": round(avg_seconds_per_page, 4) if avg_seconds_per_page is not None else None,
        "effective_seconds_per_page": round(effective_seconds_per_page, 4) if effective_seconds_per_page is not None else None,
        "throughput_files_per_sec": round(throughput_files, 4),
        "throughput_pages_per_sec": round(throughput_pages, 4),
        "peak_ram_mb": {
            "parent": parent_peak_ram_mb,
            "workers": {str(k): v for k, v in sorted(worker_peak_ram_mb.items())},
            "max_per_worker": max_worker_ram_mb,
            "sum_workers": round(sum(worker_peak_ram_mb.values()), 2) if worker_peak_ram_mb else None,
        },
        "peak_vram_mb": {
            "per_gpu": {str(k): round(v, 2) for k, v in sorted(peak_vram_mb.items())},
            "max_per_gpu": round(max_gpu_vram_mb, 2) if max_gpu_vram_mb is not None else None,
        },
    }
    perf_path.write_text(
        json.dumps(perf, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(
        f"{name}: passed={passed}, failed={len(errors)}, pages={total_pages}, "
        f"avg={average_seconds:.2f}s/file "
        f"({avg_seconds_per_page:.2f}s/page, wall={effective_seconds_per_page:.2f}s/page), "
        f"peak_ram={max_worker_ram_mb} MB/worker, peak_vram={max_gpu_vram_mb:.0f} MB/gpu"
    )
    return errors

In [ ]:
# input_dir = Path("/kaggle/input/datasets/cphmlhng/benchmark-01/DocLayNet_test/test")
# output_dir = Path("/kaggle/working/results/doclaynet_test")
# errors = convert_dataset("DocLayNet", input_dir, output_dir)

In [ ]:
# from shutil import make_archive
#
# output_dir = Path("/kaggle/working/results/doclaynet_test")
# archive = make_archive(
#     str(output_dir), "zip", root_dir=output_dir.parent, base_dir=output_dir.name
# )
# print(archive)

In [ ]:
input_dir = Path("/kaggle/input/datasets/cphmlhng/benchmark-01/OmniDocBench/en")
output_dir = Path("/kaggle/working/results/omnidocbench_en")
errors = convert_dataset("OmniDocBench EN", input_dir, output_dir)

In [ ]:
from shutil import make_archive

output_dir = Path("/kaggle/working/results/omnidocbench_en")
archive = make_archive(
    str(output_dir), "zip", root_dir=output_dir.parent, base_dir=output_dir.name
)
print(archive)

In [ ]:
input_dir = Path("/kaggle/input/datasets/cphmlhng/benchmark-01/OmniDocBench-JASyn/data")
output_dir = Path("/kaggle/working/results/omnidocbench_jasyn")
errors = convert_dataset("OmniDocBench JASyn", input_dir, output_dir)

In [ ]:
from shutil import make_archive

output_dir = Path("/kaggle/working/results/omnidocbench_jasyn")
archive = make_archive(
    str(output_dir), "zip", root_dir=output_dir.parent, base_dir=output_dir.name
)
print(archive)

In [ ]:
# input_dir = Path("/kaggle/input/datasets/cphmlhng/benchmark-01/pubtables-1m_test/test")
# output_dir = Path("/kaggle/working/results/pubtables_1m_test")
# errors = convert_dataset("PubTables-1M", input_dir, output_dir, limit=1000, seed=42)

In [ ]:
# from shutil import make_archive
#
# output_dir = Path("/kaggle/working/results/pubtables_1m_test")
# archive = make_archive(
#     str(output_dir), "zip", root_dir=output_dir.parent, base_dir=output_dir.name
# )
# print(archive)